In [74]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [75]:
data = pd.read_csv("data.csv")
data = data.dropna(axis=1, how='all')
X = data.drop(columns=['id', 'diagnosis'])
y = data['diagnosis']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [76]:
class NaiveBayes:
    def fit(self, X_train, y_train):
        self.classes = np.unique(y_train)
        self.priors = [len(y_train[y_train == c]) / len(y_train) for c in self.classes]
        
        self.means = [X_train[y_train == c].mean() for c in self.classes]
        self.stds = [X_train[y_train == c].std() for c in self.classes]

    def compute_likeihood(self, row, class_idx):
        likeihood = 1
        for feature in row.index:
            mean = self.means[class_idx][feature]
            std = self.stds[class_idx][feature]
            likeihood *= (1 / (np.sqrt(2 * np.pi) * std)) * np.exp((-(row[feature] - mean) ** 2) / (2 * std ** 2))
        
        return likeihood

    def predict(self, X):
        y_pred = []
        for _, row in X.iterrows():
            posterios = []
            for i in range(len(self.classes)):
                likeihood = self.compute_likeihood(row, i)
                posterios.append(likeihood * self.priors[i])

            y_pred.append(self.classes[np.argmax(posterios)])

        return np.array(y_pred)


In [77]:
nb = NaiveBayes()
nb.fit(X_train, y_train)
predictions = nb.predict(X_test)

accuracy = np.mean(predictions == y_test) * 100
print(f"Accuracy: {accuracy:.2f}%")

Accuracy: 96.49%
